# Modelagem preditiva e avaliação de classificadores (SOMATIVA01_TC.ipynb)

- **Repositório:** `machine-learning-portfolio`  
- **Tecnologias:** Scikit-Learn  
- **Arquivo Original:** `SOMATIVA01_TC.ipynb`  

---


# Somativa 01 - Turma C

## Estudante

In [40]:
# Colocar seu nome aqui (Desconto de 0,5 pontos se não colocar o nome)
# Victor Silva Camargo

### Importar bibliotecas

In [41]:
import numpy as np
import scipy.stats as st

## Conectar com o Google Drive

In [42]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


## Variáveis de ambiente para arquivos e conversão para PDF

In [43]:
# Path para pasta de DADOS
caminho_dados = '/content/gdrive/MyDrive/DADOS/ADS/'

# Path para pasta de PDF
caminho_pdf = '/content/gdrive/MyDrive/PDF/ADS'

# Nome do notebook
nome = 'SOMATIVA01_TC'

## Processos de Poisson

### Questão 1 (valor 1,0)
O tempo de atendimento de chamados em um helpdeks é uma variável aleatória exponencial com média igual a 4 minutos. O seu chamado está aguardando na fila com 2 chamados à sua frente (na fila) e mais 1 chamado sendo atendido. Qual a probabilidade de o seu chamado esperar mais do que 6 minutos para começar a ser atendido?

Você pode calcular a probabilidade de duas maneiras:<br>
(i) Para o chamado esperar mais do que 6 minutos é preciso que o chamado que está sendo atendido e mais os dois que estão na fila à sua frente demorem mais do que seis minutos.<br>
(ii) Para o chamado esperar mais do que 6 minutos é preciso que no intervalo de 6 minutos ocorra uma das seguintes possibilidades: nenhum chamado terminou de ser atendido no intervalo, ou 1 terminou, ou 2 terminaram. O terceiro deve terminar depois de 6 minutos.

Escolher uma delas para responder.

In [44]:
# Calcular e imprimir a probabilidade

x = 6
N = 3
mu = 4


resposta = 1- st.gamma.cdf(x, a=N, scale=mu)
print(resposta)

0.8088468305380582


## Fila MMm

In [45]:
import numpy as np
from math import factorial
from math import exp

class FilaMMm:
    def calculaP0(self):
        soma = 0
        for n in range(1, self.m):
            soma = soma + (((self.m * self.ro) ** n) / factorial(n))
        return 1 / (1 + ((self.m * self.ro) ** self.m) / (factorial(self.m) * (1 - self.ro)) + soma)

    def __init__(self, lb, mu, m):
        if (lb >= m*mu):
            raise ValueError('Lambda deve ser menor do que m*mu')
        self.lb = float(lb)
        self.mu = float(mu)
        self.m = m
        self.ro = lb / (m*mu)
        self.p0 = self.calculaP0()
        self.epsilon = ((self.m*self.ro)**self.m)/((1-self.ro)*factorial(self.m))*self.p0
        # valor esperado de Ns (quantidade media de tarefas servidas)
        self.E_Ns = self.m * self.ro
        # valor esperado de Nq (tamanho medio da fila)
        self.E_Nq = (self.epsilon * self.ro) / (1 - self.ro)
        # valor esperado de N 9quentidade media de tarefas no sistema)
        self.E_N = self.E_Nq + self.E_Ns
        # valor esperado de S (tempo de serviço médio)
        self.E_S = 1 / self.mu
        # valor esperado de W (tempo de espera medio na fila)
        self.E_W = self.epsilon / (self.m * self.mu * (1 - self.ro))
        # valor esperado de R (tempo de resposta medio)
        self.E_R = self.E_S + self.E_W


    def pmf_N(self, x):
        if (x<0 or not(isinstance(False, int))):
            raise ValueError('x deve ser inteiro positivo')
        if (x < self.m):
            return (((self.m * self.ro) ** x) / factorial(x)) * self.p0
        else:
            return (((self.ro ** x) * (self.m ** self.m)) / factorial(self.m)) * self.p0

    def pmf_Ns(self, x):
        if (x<0 or not(isinstance(False, int))):
            raise ValueError('x deve ser inteiro positivo')
        if (x < self.m):
            return self.pmf_N(x)
        if (x == self.m):
            return self.epsilon
        else:
            return 0.0

    def pmf_Nq(self, x):
        if (x<0 or not(isinstance(False, int))):
            raise ValueError('x deve ser inteiro positivo')
        if (x == 0):
            pNq = 0
            for i in range(0, self.m +1):
                pNq = pNq + self.pmf_N(i)
            return pNq
        else:
            return self.pmf_N(x+self.m)

    def cdf_W(self, x):
        if (x<0):
            raise ValueError('x deve ser positivo')
        return 1 - (self.epsilon*exp(-self.m*self.mu*(1-self.ro)*x))

    def cdf_R(self, x):
        if (x<0):
            raise ValueError('x deve ser positivo')
        if (self.ro != (self.m-1)/self.m):
            MULT = (self.epsilon / (1 - self.m + self.m * self.ro)) # calculado aqui para legibilidade
            p = 1 - exp(-self.mu * x) - MULT * (exp(-self.m * self.mu * (1 - self.ro) * x) - exp(-self.mu * x))
        else:
            p = 1 - exp(-self.mu * x) - (
                    self.epsilon * self.mu * x * exp(-self.mu * x))
        return p

### Questão 2 (valor 1,6)
O modelo de atendimento de uma solução Web foi projetado com base em um modelo de fila M/M/1 (1 servidor com uma fila, tempo entre chegadas consecutivas com distribuição exponencial, e tempo de serviço com distribuição exponencial). A taxa de chegada é de 0,45 mensagens por milissegundo e o tempo médio de serviço é de 2,0 milissegundos. A equipe de desempenho quer avaliar o tempo na fila das transações.

#### Questão 2.1 (valor 0,7)
Criar a fila. (valor 0,3)<br>
Calcular e imprimir o tempo médio na fila. (valor 0,2)<br>
Calcular e imprimir a probabilidade de uma tarefa permanecer mais de 10 milissegundos na fila. (valor 0,2)

In [46]:
# Criar fila
fila = FilaMMm(lb=0.45, mu=0.5, m=1)
# Calcular e imprimir o tempo médio na fila
print(fila.E_W)
# Calcular e imprimir a probabilidade de uma tarefa permanecer
#  mais do que 10 milissegundos na fila
print(1-fila.cdf_W(10))



18.000000000000004
0.5458775937413702


#### Questão 2.2 (valor 0,7)
Calcular e imprimir quantos servidores são necessários para que o tempo médio na fila seja menor do que 10 milissegundos.<br>
Dica: variar a quantidade de servidores em um laço até atender o requisito desejado.

In [47]:
# Calcular e imprimir quantos servidores são necessários para que o tempo médio na fila
# seja menor do que 10 milissegundos
while fila.E_W > 10:
    fila = FilaMMm(lb=0.45, mu=0.5, m=fila.m+1)
print(fila.m)

2


#### Questão 2.3 (valor 0,2)
Para a quantidade deservidores calculada na questão 2.2, calcular e imprimir a probabilidade de uma tarefa levar menos de 10 milissegundos para responder.

In [48]:
# Para a quantidade de servidores calculada na questão 2.2
# Calcular e imprimir a probabilidade de uma tarefa levar menos do que 10 milissegundos para ser completada
fila = FilaMMm(lb=0.45, mu=0.5, m=2)

print(fila.cdf_R(10))

0.9858570454005582


## Fila MG1

In [49]:
from math import sqrt
class FilaMG1:
    def __init__(self, lb, E_S, DP_S):
        if (lb >= 1/E_S):
            raise ValueError('Lambda deve ser menor do que 1/E_S')
        self.lb = float(lb)
        self.E_S = float(E_S)
        self.DP_S = DP_S
        self.ro = lb * E_S
        self.CV_S = DP_S / E_S

        # valor esperado de Ns (tamanho medio da fila)
        self.E_Ns = self.ro
        # valor esperado de Nq (tamanho medio da fila)
        self.E_Nq = (self.ro**2)*(1 +  self.CV_S**2) / (2*(1 - self.ro))
        # valor esperado de N quantidade media de tarefas no sistema)
        self.E_N = self.E_Nq + self.E_Ns
        # valor esperado de W (tempo médio de espera na fila)
        self.E_W = (self.ro*self.E_S) * (1 +  self.CV_S**2) / (2*(1 - self.ro))
        # valor esperado de R (tempo de resposta medio)
        self.E_R = self.E_S + self.E_W

### Questão 3 (valor 1,7)
Um servidor de autenticação recebe 15 solicitações por milissegundo. Verificou-se que distribuição do tempo entre duas chegadas consecutivas é exponencial. Verificou-se também que distribuição do tempo de serviço não é exponencial, mas a média do tempo de serviço é 0,042 milissegundos, e o desvio padrão é 0,031 milissegundos.

#### Questão 3.1 (valor 0,6)
Criar a fila. (valor 0,3)<br>
Calcular e imprimir o tempo de resposta médio (valor 0,3).

In [50]:
# Criar a fila
# Calcular e imprimir o tempo de resposta médio
fila2 = FilaMG1(lb=15, E_S=0.042, DP_S=0.031)
print(fila2.E_R)

0.0972364864864865


#### Questão 3.2 (valor 1,1)
Calcular a maior carga (lambda) a que pode ser submetida ao sistema sem que o tempo de resposta seja maior do que 0,5.<br>
Iniciar a carga em 15 e variá-la em um laço até atender o requisito desejado. (valor 0,8) <br>
Imprimir a maior carga encontrada e o tempo de resposta correspondente. (valor 0,3)

In [51]:
E_S = 0.042
DP_S = 0.031

lb_atual = 15.0
ultima_carga_valida = lb_atual
ultimo_tempo_resposta = None

while True:
    if lb_atual >= 1 / E_S:
        print("Limite teórico de estabilidade atingido.")
        break

    fila_teste = FilaMG1(lb=lb_atual, E_S=E_S, DP_S=DP_S)

    if fila_teste.E_R > 0.5:
        print(f"Limite Requisito com lb = {lb_atual:.2f} (E_R = {fila_teste.E_R:.4f} ms)")
        break

    ultima_carga_valida = lb_atual
    ultimo_tempo_resposta = fila_teste.E_R

    lb_atual += 0.1

print("-" * 50)
print(f"Maior λ válido: {ultima_carga_valida:.2f}")
print(f"Tempo de resposta: {ultimo_tempo_resposta:.4f} ms")


Limite Requisito com lb = 22.30 (E_R = 0.5212 ms)
--------------------------------------------------
Maior λ válido: 22.20
Tempo de resposta: 0.4894 ms


## Rede de filas

## Medidas Operacionais
* $T$: Tempo total de observação
* $M$: Número de dispositivos no sistema
* $C$: Quantidade de chegadas ao sistema no tempo T
* $C_{i}$: Quantidade de chegadas ao subsistema i no tempo T
* $P$: Quantidade de partidas do sistema (solicitações tratadas) no tempo de observação
* $P_{i}$: Quantidade de partidas do subsistema i (solicitações tratadas) no tempo de observação
* $\lambda$: Taxa de chegada no sistema
* $\lambda_{i}$: Taxa de chegada no subsistema i                
* $\mu$: Vazão do sistema
* $V$: Vazão do subsistema i
* $D$: Demanda - tempo no sistema para atender uma tarefa (incluindo todas as visitas)
* $D_{i}$: Demanda - tempo no subsistema i para atender uma tarefa (incluindo todas as visitas)
* $V_{i}$: Número de visitas ao subsistema i necessárias para atender uma tarefa
* $N$: Número médio de tarefas no sistema
* $N_{i}$: Número médio de tarefas no subsistema i
* $R$: Tempo médio de resposta do sistema
* $R_{i}$: Tempo médio de resposta do subsistema i
* $S$: Tempo médio de serviço no sistema
* $S_{i}$: Tempo médio de serviço no subsistema i
* $B$: Tempo que o sistema permanece ocupado
* $B_{i}$: Tempo que o subsistema i permanece ocupado
* $U$: Taxa de utilização do sistema
* $U_{i}$: Taxa de utilização do subsistema i

## Relações Operacionais
* $\lambda_{i}=C_{i}/T$
* $\mu_{i}=P_{i}/T$
* $U_{i}=B_{i}/T$
* $S_{i}=B_{i}/P_{i}$
* $V_{i}=P_{i}/P$
* $\mu_{i}=\mu/V_{i}$
* $D=\sum_{}^{}D_{i}$
* $D_{i}=S_{i}\cdot V_{i}$
* $U_{i}=\mu_{i}\cdot S_{i}=\mu\cdot S_{i}\cdot V_{i}=\mu \cdot D_{i}$
* $N_{i}= U_{i} / (1 - U_{i})$
* $N_{i}= \mu_{i} \cdot R_{i}$
* $N=\sum_{}^{}N_{i}$
* $R_{i}= S_{i} / (1 - U_{i})$
* $R=\sum_{}^{} R_{i}\cdot V_{i}$

### Questão 4 (valor 1,2)
O desempenho de um servidor web está sendo avaliado. A taxa de utilização observada foi de 40%. O tempo de resposta médio das transações foi de 0,18 milissegundos.<br>
* a) Calcular o número medio de transações executadas nesse período de observação. (valor 0,3)
* b) Calcular a vazão. (valor 0,3)
* c) Calcular o tempo de serviço médio. (valor 0,3)
* d) Calcular o tempo de resposta se a taxa de utilização subir para 70%. (valor 0,3)


Dados
* $U = 0.4$
* $R = 0.18$<br>



In [52]:
# Calcular e imprimir o número médio de tarefas no sistema (valor 0,3)
U = 0.4
R = 0.18
N = U / (1 - U)
print(N)
# Calcular e imprimir a vazão (valor 0,3)
u = N / R
print(u)
# Calcular e imprimir o tempo de serviço médio (valor 0,3)
S = U / u
print (S)
# Recalcular e imprimir o tempo de resposta para utilização de 70% (valor 0,3)
U2 = 0.7
R2 = S/(1 - U2)
print(R2)


0.6666666666666667
3.703703703703704
0.108
0.35999999999999993


## Entrega
* Colocar o nome.
* Resolver as quetões.
* Exportar para pdf.
* Conferir se o pdf contém todas as respostas legíveis.
* Alterar no nome do arquivo para seu_nome.pdf.
* Submeter no AVA.
* Entrega individual.

In [53]:
#%%
##################################################################
######    Script para geração de pdf    ##########################
##################################################################

##################################################################
#######    Instalar bibliotecas         ##########################
##################################################################
!apt-get install texlive texlive-xetex texlive-latex-extra pandoc
!pip install pypandoc

##################################################################
#######   Copiar notebook para ambiente collab   #################
##################################################################
nome_notebook = nome +'.ipynb'
path_notebook = '/content/gdrive/MyDrive/Colab Notebooks/' + nome_notebook
!cp "{path_notebook}" ./

##################################################################
#######    Converter para pdf          ###########################
##################################################################
nome_notebook = nome +'.ipynb'
!jupyter nbconvert --to PDF "{nome_notebook}"

##################################################################
#######      Salvar no drive              ########################
##################################################################
nome_pdf = nome +'.pdf'
caminho_local = '/content/'+ nome_pdf
!cp "{caminho_local}" "{caminho_pdf}"

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pandoc is already the newest version (2.9.2.1-3ubuntu2).
pandoc set to manually installed.
The following additional packages will be installed:
  dvisvgm fonts-droid-fallback fonts-lato fonts-lmodern fonts-noto-mono
  fonts-texgyre fonts-urw-base35 libapache-pom-java libcommons-logging-java
  libcommons-parent-java libfontbox-java libgs9 libgs9-common libidn12
  libijs-0.35 libjbig2dec0 libkpathsea6 libpdfbox-java libptexenc1 libruby3.0
  libsynctex2 libteckit0 libtexlua53 libtexluajit2 libwoff1 libzzip-0-13
  lmodern poppler-data preview-latex-style rake ruby ruby-net-telnet
  ruby-rubygems ruby-webrick ruby-xmlrpc ruby3.0 rubygems-integration t1utils
  teckit tex-common tex-gyre texlive-base texlive-binaries
  texlive-fonts-recommended texlive-latex-base texlive-latex-recommended
  texlive-pictures texlive-plain-generic tipa xfonts-encodings xfonts-utils
Suggested packages:
  fonts-noto f